In [1]:
import torch
import numpy as np

from botorch.models.gp_regression import SingleTaskGP
from botorch.models.deterministic import GenericDeterministicModel
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.models.model import ModelList

from gpytorch.mlls import ExactMarginalLogLikelihood

from botorch import fit_gpytorch_mll
from botorch.optim.optimize import optimize_acqf_discrete
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.utils.multi_objective.box_decompositions.dominated import DominatedPartitioning

import time
import warnings
from botorch.exceptions.warnings import NumericsWarning
warnings.filterwarnings("ignore", category=NumericsWarning)
warnings.filterwarnings("ignore", category = RuntimeWarning)

tkwargs = {
    "dtype": torch.double,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

# Define default parameters for the BO loop
REFERENCE = torch.tensor([-0.1, 0.0], **tkwargs)
BATCH_SIZE = 10
N_ITER = 10
MAX_CONC = 8
STEP = 0.5

# Define default hyperparams for the acqf optimization routine
DEFAULT_PARAMS = {
    "max_batch_size": 128
}

In [2]:
def loadData():
    data = np.concatenate([
        np.load('data/data_initial.npy'),
        np.load('data/data_iter1.npy'),
        np.load('data/data_iter2.npy')
    ])
    return torch.tensor(data).to(**tkwargs)

def loadChoices():
    return torch.load(f'data/combinations-{MAX_CONC}M.pt').to(**tkwargs)

from mlp import MLP

# Load the surrogate model
mdl = MLP().to(**tkwargs)
mdl.load_state_dict(torch.load('models/surrogate.pth', weights_only=True))
mdl.eval()

def func1(X):
    # Sum the individual concentrations to a total concentration
    return X.sum(axis = 1).reshape(-1, 1)

def func2(X):
    # Utilize the MLP surrogate to predict viability
    return mdl.predict(X)

def multi_objective(X):
    return torch.cat([func1(X), func2(X)], dim=1)

In [3]:
def initialize_model(train_X, train_Y):
    # Define deterministic model (on total concentration, doesnt require train_Y though)
    deterministic_model = GenericDeterministicModel(lambda x: x.sum(dim=-1, keepdim=True))
    # Define GP model (on viability)
    gp_model = SingleTaskGP(
        train_X,
        train_Y[:,1].unsqueeze(-1),
        input_transform = Normalize(train_X.shape[-1]),
        outcome_transform = Standardize(m=1)
    )
    model = ModelList(deterministic_model, gp_model)
    mll = ExactMarginalLogLikelihood(model.models[1].likelihood, model.models[1])
    return mll, model

def step_mobo(model, train_X, choices, max_batch_size):
    acq = qLogNoisyExpectedHypervolumeImprovement(
        model = model,
        ref_point = REFERENCE,
        X_baseline = train_X,
        prune_baseline = True
    )
    candidates, _ = optimize_acqf_discrete(
        acq_function = acq,
        q = BATCH_SIZE,
        choices = choices,
        X_avoid = train_X,
        unique = True,
        max_batch_size = max_batch_size
    )
    return candidates

def run_bo():
    init_X = loadData()[:,:-2]
    init_Y = loadData()[:,-2:]

    choices = loadChoices()

    mll, model = initialize_model(init_X, init_Y)

    train_X, train_Y = init_X.clone(), init_Y.clone()

    hvs = [DominatedPartitioning(REFERENCE, init_Y).compute_hypervolume().item()]
    times = [0.0]
    candidates_list = [init_X.cpu().tolist()]

    print(f"Iteration {0} | HV = {hvs[0]:.5f}, t = {times[0]}s")

    for i in range(1, N_ITER+1):
        start = time.time()
        fit_gpytorch_mll(mll)
        candidates = step_mobo(model, train_X, choices, **DEFAULT_PARAMS)
        end = time.time()

        # Empty the cuda cache to save memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Add the candidates to the training set
        new_X = candidates.detach()
        new_Y = multi_objective(new_X)
        train_X = torch.cat([train_X, new_X], dim=0)
        train_Y = torch.cat([train_Y, new_Y], dim=0)

        # Compute the hypervolume
        hv = DominatedPartitioning(ref_point=REFERENCE, Y = train_Y).compute_hypervolume().item()

        # Save all relevant data to lists
        hvs.append(hv)
        times.append(end-start)
        candidates_list.append(candidates.cpu().tolist())

        mll, model = initialize_model(train_X, train_Y)

        print(f"Iteration {i} | HV = {hv:.5f}, t = {end-start:.2f}s")

    return hvs, times, candidates_list

In [4]:
BATCH_SIZE = 10
N_ITER = 10
MAX_CONC = 8
DEFAULT_PARAMS = {
    "max_batch_size": 1024
}

with torch.random.fork_rng(devices = [0]):
    hvs, times, candidates_list = run_bo()

Iteration 0 | HV = 594.93836, t = 0.0s


/home/dan/Research/cpa-bayesian-opt/py31/lib/python3.11/site-packages/torch/random.py:187: UserWarning: CUDA reports that you have 4 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  warnings.warn(message)


Iteration 1 | HV = 686.74332, t = 689.47s
Iteration 2 | HV = 700.29342, t = 776.59s
Iteration 3 | HV = 713.64535, t = 798.05s
Iteration 4 | HV = 718.64450, t = 846.78s
Iteration 5 | HV = 722.21772, t = 834.41s
Iteration 6 | HV = 722.81971, t = 905.19s
Iteration 7 | HV = 727.36288, t = 937.79s
Iteration 8 | HV = 735.11465, t = 1117.28s
Iteration 9 | HV = 736.20437, t = 1236.19s
Iteration 10 | HV = 736.20437, t = 1346.21s


In [15]:
import plotter

plotter.generate_gif(candidates_list, MAX_CONC, 'exhaustive-search-8M.gif')

GIF saved as exhaustive-search-8M.gif
